In [34]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

### dataset
print (f"\n ----- Data Set ----")
data = np.array([["A","B","C","D","E","F","G","H","I"],
                [7, 7, 3, 1, 3, 4, 6, 2, 3],
                [50, 70, 65, 60, 45, 54, 55, 59, 75],
                ["bad","bad","good","good","bad","good","good","bad","bad"]
                ])

print(data)

###### ----CLASSIFICATION PAR KNN -----#####

# 1. Données d'entraînement (A, B, C, D, E, F, G, H, I)
X_train = np.array([[7, 50], [7, 70], [3, 65], [1, 60], [3, 45],[4, 54],[6, 55],[2, 59],[3, 75]]) # les featurs:[Cigarettes , Weight]
y_train = np.array([0, 0, 1, 1, 0, 1, 1, 0, 0]) # 0: Bad , 1: Good 

# 2. Nouveau point de test J
x_test = np.array([8, 50])

# --- STANDARDISATION ---
# On calcule moyenne et écart-type sur la base de référence
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

X_train_scaled = (X_train - mean) / std
x_test_scaled = (x_test - mean) / std

# --- KNN FROM SCRATCH ---
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2)**2))

def predict_knn(X, y, point, k=K):
    # Calcul des distances entre F et tous les points A,B,C,D,E,F,G,H,I
    distances = [euclidean_distance(point, x) for x in X]
    
    # Trouver les indices des k plus proches voisins
    k_indices = np.argsort(distances)[:k]
    k_nearest_labels = [y[i] for i in k_indices]
    
    # Affichage des distances pour l'analyse
    print(f"\n------ Les Distances -----")
    print(f"Distances standardisées de J vers [A, B, C, D, E, F, G, H, I] : \n{np.round(distances, 2)}")
    print(f"Les {k} voisins les plus proches de J sont de classes : {k_nearest_labels}")
    
    # Vote majoritaire
    return Counter(k_nearest_labels).most_common(1)[0][0]

# --- RÉSULTAT ---
k = 3
resultat = predict_knn(X_train_scaled, y_train, x_test_scaled, k=K)

classe_finale = "Good" if resultat == 1 else "Bad"
print(f"\n--- RÉSULTAT FINAL ---")
print(f" ***la classification ***")
print(f"Le point J (8, 50) appartient à la classe : {classe_finale}")
 

###### ----REGRESSION PAR KNN -----#####
 ## au niveau de la regression on va utiliser des continues donc pour ce la on va chonger les classes ( bad / good) par des valeurs numerique
 # Scores de santé (au lieu de Bad/Good) 
y_train = np.array([10, 25, 80, 95, 15, 50, 45, 30, 90])


def knn_regression(X, y, point, k=3):
    distances = [np.sqrt(np.sum((point - x)**2)) for x in X]
    k_indices = np.argsort(distances)[:k]
    
    # MOYENNE au lieu de VOTE
    k_nearest_values = [y[i] for i in k_indices]
    return np.mean(k_nearest_values)

prediction = knn_regression(X_train_scaled, y_train, x_test_scaled, k=K)
print(f"\n *** la regression ***")
print(f"Le score de santé prédit pour J est : {prediction:.2f}%")





 ----- Data Set ----
[['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I']
 ['7' '7' '3' '1' '3' '4' '6' '2' '3']
 ['50' '70' '65' '60' '45' '54' '55' '59' '75']
 ['bad' 'bad' 'good' 'good' 'bad' 'good' 'good' 'bad' 'bad']]

------ Les Distances -----
Distances standardisées de J vers [A, B, C, D, E, F, G, H, I] : 
[0.49 2.27 2.95 3.58 2.5  2.   1.12 3.09 3.69]
Les 3 voisins les plus proches de J sont de classes : [np.int64(0), np.int64(1), np.int64(1)]

--- RÉSULTAT FINAL ---
 ***la classification ***
Le point J (8, 50) appartient à la classe : Good

 *** la regression ***
Le score de santé prédit pour J est : 35.00%


In [36]:
import numpy as np

# 1. Dataset (A à I)
data = np.array([
    ["A","B","C","D","E","F","G","H","I"],
    [7, 7, 3, 1, 3, 4, 6, 2, 3],
    [50, 70, 65, 60, 45, 54, 55, 59, 75],
    ["bad","bad","good","good","bad","good","good","bad","bad"]
])

# Préparation des données
X = data[1:3, :].T.astype(float)
# Le SVM utilise souvent -1 et 1 au lieu de 0 et 1
y = np.where(data[3, :] == "good", 1, -1)

# Standardisation (Indispensable pour le SVM)
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)
X_scaled = (X - mean) / std

class SVM_from_scratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param # Paramètre de régularisation
        self.n_iters = n_iters
        self.w = None # Poids
        self.b = None # Biais

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # Condition du SVM : y_i * (w * x_i - b) >= 1
                condition = y[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    # On ne met à jour que la régularisation
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # On met à jour les poids et le biais (Hinge Loss)
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y[idx]))
                    self.b -= self.lr * y[idx]

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)

# --- Exécution ---
model = SVM_from_scratch()
model.fit(X_scaled, y)

# Test sur le point J (8, 50)
point_f = np.array([8, 50])
point_f_scaled = (point_f - mean) / std
prediction = model.predict(point_f_scaled)

resultat = "Good" if prediction == 1 else "Bad"
print(f"Poids (w) : {model.w}")
print(f"Biais (b) : {model.b}")
print(f"\nRésultat SVM pour le point J (8, 50) : {resultat}")

Poids (w) : [-5.94247855e-01 -3.99843943e-05]
Biais (b) : 0.1310000000000001

Résultat SVM pour le point J (8, 50) : Bad
